# 15 · E-MNIST: dimensión que corre d(E) sobre una malla de β hasta la fisión

Mide la dimensión **dependiente de la escala** de E-MNIST (split `balanced`), como en Kaluza–Klein: a energía baja el espacio se ve de pocas dimensiones y a energía alta aparecen más. Aquí β fija la energía más alta que el kernel gaussiano puede resolver (E ≈ 4β): un β mayor ve más, y la dimensión local d(E) debe ser **la misma curva** donde los rangos fiables de distintos β se solapan. Ese solape es la prueba de invarianza.

**Protocolo.** Con K_ij = exp(−β‖x_i − x_j‖²) y P = D⁻¹K (`alpha = 0`), `dimension.py` diagonaliza el operador simétrico conjugado en float64 y toma todos los autovalores λ_j. Las energías son E_j = 4β(1 − λ_j), con E_0 = 0. La malla es logarítmica, de 0,1·β_base a β = 10 (32 valores, en unidades de la entrada estandarizada). Se resuelven los β **en orden creciente** y el barrido se detiene en el primero que **fisiona** (hueco 1 − λ_1 ≤ 1e-8, el grafo se parte en componentes): los β mayores no se resuelven. β* es el mayor β antes de la fisión.

**Tres estimadores.**
- **A (suma del kernel).** S(β) = Σ_{i≠j} K_ij (sin la diagonal) y d = −2 dlog S / dlog β, en E = 4β. No necesita autovalores, así que se evalúa en una malla más densa y más amplia (`a_points = 64`, desde 0,01 × β_base, para ver la caída a d → 0) con una sola pasada por teselas.
- **B (traza del calor).** Z(τ) = Σ_j exp(−τE_j) con τ ∈ [1/(4β), 10³/(4β)] y d = −2 dlog Z / dlog τ, en E = 1/τ.
- **C (conteo de Weyl).** N(E) = #{j ≥ 1: E_j ≤ E} y d = 2 dlog N / dlog E, suavizada con una media móvil de 5 puntos.

**Rango fiable.** Para cada β se conservan solo E ≤ β (parámetro `trust`). Como se usa el espectro completo no hace falta la regla E_k/3 de truncamiento. Para β pequeños todas las energías se agrupan cerca de 4β y no hay niveles fiables: esos β solo aportan a través de A.

**Forma esperada.** d(E) cae a 0 en las energías más bajas, forma mesetas con escalones hacia arriba (la dimensión de la variedad a cada escala) y sube o se desploma cerca de β*. Calibración: la dimensión intrínseca publicada para los dígitos es ~10; las mesetas deberían acercarse a ese orden, pero **no hay un valor medido todavía**.

**Figuras.** `dimension_n`: d frente a n = N(E), el número de niveles por debajo de E (sin normalizar por N), una curva por β (C continua, B discontinua), como en el boceto. `dimension_E`: d frente a E con A, B y C en colores distintos.

**Por qué float64 denso en CPU a N = 19 200 y no FlashDiffusion.** Flash solo ofrece productos de un vector: calcular los k primeros modos a este N cuesta ~22 min por β, y el espectro completo es inviable. El denso en CPU da **todos** los autovalores en ~9 min por β (medido en el notebook 14). Los autovalores se calculan desde β_base (por debajo los niveles caen todos fuera del rango confiable E ≲ β, y solo aporta A). Estimación del total: ~2–2,5 h (sin medir), porque los β por encima de β* no se resuelven. Memoria: una matriz N × N en float64 (≈ 3 GB).

**Datos:** adjunta el Kaggle Dataset **crawford/emnist** (el workflow de GitHub lo adjunta solo y lanza este notebook como sesión de CPU, sin gastar cuota de GPU).

## Preparación

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
ROOT = Path('/kaggle/working/dmap_vae_kaggle')
INPUT = Path('/kaggle/input')
NB13 = 'dndr-13-emnist-kaggle'   # a kernel output that may hold its own copy of the code: never a package source
CODE = ['spectral.py', 'vae.py', 'data.py', 'experiment.py', 'launch.py', 'plots.py',
        'checks.py', 'prepare_alanine.py', 'generate_angles.py', 'SOURCE.json',
        'geometry.py', 'beta_sweep.py', 'config_beta_sweep.json',
        'config_emnist.json', 'dimension.py', 'spectrum.py', 'config_dimension_emnist.json']

# Refresh the code on EVERY run (a partial copy left in /kaggle/working is never trusted).
sources = sorted({q.parent for q in INPUT.rglob('launch.py')
                  if (q.parent / 'spectral.py').exists() and NB13 not in q.parts})
archives = sorted(a for a in INPUT.rglob('dmap_vae_kaggle*.zip') if NB13 not in a.parts)
if len(sources) > 1:
    raise FileNotFoundError(f'Hay varias copias del paquete en /kaggle/input: {sources}. Deja solo una.')
ROOT.mkdir(parents=True, exist_ok=True)
if sources:
    shutil.copytree(sources[0], ROOT, dirs_exist_ok=True)
elif len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(ROOT.parent)
else:
    raise FileNotFoundError('Añade el paquete dmap_vae_kaggle como Kaggle Dataset; debe haber una sola copia.')
missing_code = [f for f in CODE if not (ROOT / f).exists()]
assert not missing_code, f'Faltan archivos del paquete: {missing_code}'

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import importlib.util
missing = [m for m in ['numpy','scipy','pandas','matplotlib']
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
print('CPU float64, hilos:', torch.get_num_threads())
(ROOT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
from IPython.display import display, Image

## Datos E-MNIST

Se prepara la caché **una sola vez** (la escritura es atómica).

In [ ]:
from data import ensure_emnist
EMNIST = ensure_emnist('/kaggle/working/emnist_balanced.npz', 'balanced')
import numpy as np
with np.load(EMNIST) as a:
    print({k: a[k].shape for k in a.files}, 'clases:', int(a['y_train'].max()) + 1)

## Espectro completo para N = 4800, 9600 y 19 200

Una ejecución de `spectrum.py` por tamaño (`k: "all"`, float64 en CPU, todas las imágenes de la submuestra como anclas). Cada tamaño escribe `spectrum.npz`, `spectrum_meta.json` y las tres figuras habituales en `spectrum_emnist_full/N<N>/`.

## Dimensión que corre d(E)

Una sola ejecución de `dimension.py` con la configuración `config_dimension_emnist.json`. Imprime β_base, la malla, y por cada β resuelto el hueco, el número de niveles fiables y los segundos. Escribe `dimension.npz`, `dimension_meta.json` y las dos figuras en `dimension_emnist/`.

In [ ]:
cfg = json.loads((ROOT / 'config_dimension_emnist.json').read_text())
cfg['data'] = str(EMNIST)
OUT = Path(cfg['out'])
config = ROOT / 'active_dimension.json'
config.write_text(json.dumps(cfg, indent=2))
started = time.perf_counter()
subprocess.run([sys.executable, str(ROOT / 'dimension.py'), str(config)], check=True)
print(f'dimension.py: {(time.perf_counter() - started) / 60:.1f} min')

## Figuras

`dimension_n`: d frente a n = N(E), una curva por β (color por log β; C continua, B discontinua) sobre las energías fiables. `dimension_E`: d frente a E, con A (puntos, uno por β), B y C (una línea fina por β). Donde los rangos fiables se solapan, las curvas de distintos β deben coincidir.

In [ ]:
display(Image(filename=str(OUT / 'dimension_n.png')))
display(Image(filename=str(OUT / 'dimension_E.png')))

## β* y resumen por β

β* es el mayor β con hueco > 1e-8. Para cada β resuelto: hueco, niveles fiables (E ≤ β) y mediana de d (estimador C) sobre su rango fiable.

In [ ]:
meta = json.loads((OUT / 'dimension_meta.json').read_text())
print(f"beta* = {meta['beta_star']:.4g}, N = {meta['n_anchors']}")
print(f"{'beta':>10}{'gap':>12}{'trusted':>9}{'median d (C)':>14}")
for r in meta['betas']:
    d = r['median_c_d']
    print(f"{r['beta']:>10.4g}{r['gap']:>12.3e}{r['trusted_levels']:>9d}" + (f"{d:>14.2f}" if d is not None else f"{'-':>14}"))

## Archivos guardados

`dimension_emnist/`: `dimension.npz` (β resueltos, energías, huecos, β*, suma del kernel de A), `dimension_meta.json` (configuración, β* y el resumen por β), `dimension_n` y `dimension_E` (.png/.pdf).